In [53]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/abderhmanahmed/therapsit-sound-test-1/TherapistAI_test1.mp3
/kaggle/input/datasets/abderhmanahmed/cbt-guide/therapists_guide_to_brief_cbtmanual.pdf


In [54]:
!pip install faiss-cpu langchain langchain-community langchain-core langchain-huggingface pypdf sentence-transformers transformers==4.52.4 torch

In [55]:
!pip install transformers==4.52.4  langchain-classic

In [56]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [57]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer_ministrali= AutoTokenizer.from_pretrained(model_name)
model_minisralai = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [100]:
def generate_text(prompt, max_new_tokens=1000):
    inputs = tokenizer_ministrali(prompt, return_tensors="pt").to(model_minisralai.device)

    outputs = model_minisralai.generate(
        **inputs,
        max_new_tokens=max_new_tokens,   # ✅ only generate new text
        do_sample=False,                 # 🔥 deterministic
        temperature=0.0,                 # 🔥 no randomness
        pad_token_id=tokenizer_ministrali.eos_token_id
    )

    full_output = tokenizer_ministrali.decode(outputs[0], skip_special_tokens=True)

    # ✅ Remove prompt safely
    if full_output.startswith(prompt):
        generated = full_output[len(prompt):].strip()
    else:
        generated = full_output.strip()

    return generated

In [59]:
from langchain_core.language_models.llms import LLM
from typing import Any

class CustomHFLLM(LLM):
    def _call(self, prompt: str, stop: Any = None) -> str:
        return generate_text(prompt)
    
    @property
    def _llm_type(self) -> str:
        return "custom_huggingface"

llm = CustomHFLLM()

In [60]:
pdf_path = "/kaggle/input/datasets/abderhmanahmed/cbt-guide/therapists_guide_to_brief_cbtmanual.pdf"
loader = PyPDFLoader(pdf_path)
documents = loader.load()

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)

embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding = HuggingFaceEmbeddings(model_name=embedding_model_name)
vectordb = FAISS.from_documents(chunks, embedding)

In [61]:
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.prompts import PromptTemplate
import re

risklevel_schema= ResponseSchema(
    name="risk_level",
    description="A level to indicate the risk the user input is in from Low or medium or high."
)

selfharm_schema=ResponseSchema(
    name="self_harm",
    description="A boolean (ANSWER Only in[True or False]) which indicate weither the user input has intended to harm himself or already hurt himself or not"
)

suicidalintent_schema= ResponseSchema(
    name="suicidal_intent",
    description="A Boolean(ANSWER Only in[True or False]) to indicate weither the user input has the intended to suicide or not."
)

hopelessness_schema= ResponseSchema(
    name="hopelessness",
    description="A Boolean(ANSWER Only in[True or False]) to indicate weither the user input has hopelessness or not."
)

abuse_schema= ResponseSchema(
    name="abuse",
    description="A Boolean(ANSWER Only in[True or False]) to indicate weither the user is abused or not."
)
response_schemas=[risklevel_schema,selfharm_schema,suicidalintent_schema,hopelessness_schema,abuse_schema]
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()

In [62]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain

import json
safety_prompt = PromptTemplate(
    input_variables=["input","format_instructions"],
    template="""
You are a mental health safety classifier.

Respond ONLY in JSON format as follows,You MUST include ALL fields in the JSON.
Do not omit any keys:
{format_instructions}

User message:
{input}

Answer:
"""
)

safety_chain = LLMChain(
    llm=llm,
    prompt=safety_prompt
)

In [63]:
cbt_prompt = PromptTemplate(
    input_variables=["input"],
    template="""
You are a CBT analyzer.

Extract ONLY JSON:
{{
  "emotion": "",
  "intensity": 0.0,
  "cognitive_distortion": "",
  "trigger": "",
  "automatic_thought": "",
  "core_belief": "",
  "behavior": ""
}}

User message:
{input}

Answer:
"""
)

cbt_chain = LLMChain(
    llm=llm,
    prompt=cbt_prompt
)

In [64]:
CBT_Rag_prompt= PromptTemplate(
    input_variables=["input", "cbt"],
    template="""
    You are a CBT (Cognitive Behavioral Therapy) summarization assistant.

Your task is to create a concise, structured summary that captures the psychological meaning of the conversation so it can be compared with CBT knowledge chunks.

Focus on:
- Core problem (situation)
- Emotions
- Negative thoughts / cognitive distortions
- Behaviors
- Key CBT concepts (if present)

Return ONLY a short structured summary in plain text.

---

CBT Data:
{cbt}

User Input:
{input}


Summary:
- Problem:
- Emotions:
- Thoughts:
- Behaviors:
- CBT Concepts:


Answer:
"""
)

CBT_Rag_chain = LLMChain(llm=llm, prompt=CBT_Rag_prompt)

In [95]:
def router(safety_result):
    risk = safety_result["risk_level"]
    #flags = safety_result["flags"]

    if risk == "High" or safety_result["self_harm"]=="True" or safety_result["suicidal_intent"]=="True":
        return "crisis"

    if risk == "Medium" or safety_result["hopelessness"]=="True" or safety_result["abuse"]=="True":
        return "support"

    return "full"

In [66]:
crisis_prompt = PromptTemplate(
    input_variables=["input","context","safety"],
    template="""
You are a supportive mental health assistant.

The user may be in distress.



User message:
{input}

Respond with empathy and encourage reaching out for help.
always add that the Egyption emergancy number is 112,Befrienders Cairo (Hotline): 7621602
General Secretariat of Mental Health: 08008880700 or 0220816831

Respond:
"""
)

crisis_chain = LLMChain(llm=llm, prompt=crisis_prompt)

In [67]:
support_prompt = PromptTemplate(
    input_variables=["input", "cbt","context"],
    template="""
You are a gentle CBT assistant.

Be supportive, do NOT challenge strongly.

CBT data:
{cbt}

User:
{input}

Use the following context to respond

Context:
{context}

Respond gently with validation + light reframing.

Respond:
"""
)

support_chain = LLMChain(llm=llm, prompt=support_prompt)

In [68]:
full_prompt = PromptTemplate(
    input_variables=["input", "cbt","context"],
    template="""
You are a CBT therapist assistant.

Use structured CBT:
- identify distortion
- challenge thought
- reframe belief
- suggest action



CBT data:
{cbt}

User:
{input}


Use the following context to respond

Context:
{context}

Respond:
"""
)

full_chain = LLMChain(llm=llm, prompt=full_prompt)

In [77]:
def extract_json_block(text):
    pattern = r'```json\s*(.*?)\s*```'
    matches = re.findall(pattern, text, re.DOTALL)

    return f"```json\n{matches[-1]}\n```"

In [101]:
def run_pipeline(user_input):


    # 1. Safety
    safety_raw = safety_chain.run(
        input=user_input,
        format_instructions=format_instructions)
    safety = output_parser.parse(extract_json_block(safety_raw))

    # 2. CBT
    cbt_raw = cbt_chain.run(user_input)
    cbt = json.loads(cbt_raw)

    cbt_rag_result=CBT_Rag_chain.run(
        input=user_input,
        cbt=cbt_raw
    )
    
    docs = vectordb.similarity_search(cbt_rag_result, k=2)
    context = "\n\n".join([doc.page_content for doc in docs])
    
    
    # 3. Route decision
    mode = router(safety)

    # 4. Select chain
    if mode == "crisis":
        response = crisis_chain.run(
            input=user_input,
            context=context
        )

    elif mode == "support":
        response = support_chain.run(
            input=user_input,
            cbt=cbt_raw,
            context=context
        )

    else:
        response = full_chain.run(
            input=user_input,
            cbt=cbt_raw,
            context=context
        )

    return {
        "mode": mode,
        "safety": safety,
        "cbt": cbt,
        "response": response
    }

In [102]:
userInput="I am autistic and afraid to talk to people."

result=run_pipeline(userInput)
print(result)

{'mode': 'full', 'safety': {'risk_level': 'Low', 'self_harm': 'False', 'suicidal_intent': 'False', 'hopelessness': 'False', 'abuse': 'False'}, 'cbt': {'emotion': 'fear', 'intensity': 8.0, 'cognitive_distortion': 'overgeneralization, social isolation', 'trigger': 'interacting with people', 'automatic_thought': "I'll make a fool of myself", 'core_belief': 'I am not good enough to socialize', 'behavior': 'avoidance'}, 'response': 'Based on the information you\'ve provided, it seems that you\'re experiencing fear and avoidance when it comes to interacting with people. This is a common experience for many individuals, especially those with autism. Let\'s break down the different elements of this situation using the CBT skills we\'ve learned.\n\nFirst, let\'s identify the cognitive distortion that\'s contributing to your fear. In this case, it appears to be overgeneralization and social isolation. Overgeneralization occurs when we take one negative experience and apply it to all situations, 

In [103]:
print(result["response"])

Based on the information you've provided, it seems that you're experiencing fear and avoidance when it comes to interacting with people. This is a common experience for many individuals, especially those with autism. Let's break down the different elements of this situation using the CBT skills we've learned.

First, let's identify the cognitive distortion that's contributing to your fear. In this case, it appears to be overgeneralization and social isolation. Overgeneralization occurs when we take one negative experience and apply it to all situations, while social isolation is the belief that we are alone in our experiences and that no one else can understand or relate to us.

Next, let's challenge these thoughts. Overgeneralization is not a realistic way to view the world. Just because one interaction with someone didn't go well, it doesn't mean that all interactions will be the same. And as for social isolation, remember that everyone experiences difficulties in social situations f